# 検索・ランキングを評価する (Recall@k、MRR、nDCG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_search/ranking_metrics.ipynb)

NLP クックブック 19-10 ／ 検索

検索システムやリランカーが、正解の文書をどれくらい上位に出せているかを測りたい。

In [1]:
import numpy as np

def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / len(relevant)

def reciprocal_rank(ranked, relevant):
    for i, doc in enumerate(ranked, 1):
        if doc in relevant:
            return 1 / i
    return 0.0

def average_precision(ranked, relevant):
    hits, total = 0, 0.0
    for i, doc in enumerate(ranked, 1):
        if doc in relevant:
            hits += 1
            total += hits / i            # 正解が出た位置での適合率
    return total / len(relevant)

def ndcg_at_k(ranked, gains, k):
    """gains: {文書 ID: 関連度 (0, 1, 2, ...)}。上位ほど大きい重みで関連度を足し、理想的な順位での値で割る"""
    dcg = sum(gains.get(doc, 0) / np.log2(i + 1) for i, doc in enumerate(ranked[:k], 1))
    ideal = sum(g / np.log2(i + 1) for i, g in enumerate(sorted(gains.values(), reverse=True)[:k], 1))
    return float(dcg / ideal) if ideal else 0.0

# クエリごとの (システムの出力順位, 関連度付きの正解)
runs = {
    "q1": (["d3", "d1", "d7", "d2", "d9"], {"d1": 2, "d2": 1}),
    "q2": (["d5", "d4", "d8", "d6", "d1"], {"d5": 2}),
    "q3": (["d2", "d9", "d4", "d3", "d6"], {"d7": 2, "d6": 1}),
}
rows = []
for q, (ranked, gains) in runs.items():
    rel = set(gains)
    rows.append([recall_at_k(ranked, rel, 1), recall_at_k(ranked, rel, 5), reciprocal_rank(ranked, rel), average_precision(ranked, rel), ndcg_at_k(ranked, gains, 5)])
    print(q, [round(x, 3) for x in rows[-1]])
print("平均: Recall@1 {:.3f}  Recall@5 {:.3f}  MRR {:.3f}  MAP {:.3f}  nDCG@5 {:.3f}".format(*np.mean(rows, axis=0)))

q1 [0.0, 1.0, 0.5, 0.5, 0.643]
q2 [1.0, 1.0, 1.0, 1.0, 1.0]
q3 [0.0, 0.5, 0.2, 0.1, 0.147]
平均: Recall@1 0.333  Recall@5 0.833  MRR 0.567  MAP 0.533  nDCG@5 0.597


- RAG の検索段階なら Recall@k (正解が上位 k 件に入っているか)、最終的な順位付けなら MRR や nDCG を見る。
- MRR は最初の正解の順位だけを見る。正解が複数あって段階的な関連度が付いているなら nDCG。
- 標準的な実装は `pytrec_eval` や `ranx`。論文で報告するなら、それらを使うのが安全。